# 12 · Swiss roll con volumen (intrínsecamente 3-D)

El mismo rollo del artículo (espiral r = 0.5 + 0.6 θ, W = 60, H = 10), pero **con grosor**: cada punto de la lámina se desplaza una distancia t ∈ [−T/2, T/2] a lo largo de la normal de la espiral. La carta verdadera es (s, h, t); la variedad es de dimensión 3. Con T = 2, entre vueltas consecutivas quedan 1.77 unidades libres, así que las vueltas no se tocan. La carta es exacta pero **no isométrica**: fuera de la superficie central el eje s se estira por (1 + κ t).

**Predicción (el teorema del prefijo en 3-D).** En la aproximación de caja plana [0, W] × [0, H] × [0, T] con condiciones de Neumann, los modos son cos(kπs/W) cos(lπh/H) cos(jπt/T). El primer modo que depende de h es (0, 1, 0) y el primero que depende de t es (0, 0, 1). Todo modo anterior es función de las otras coordenadas, así que **ninguna lectura afín de un prefijo más corto contiene información de ese eje**. Con índice ψ₀ = primer modo no trivial:

| eje | primer modo | índice para 60 × 10 × 2 |
|---|---|---|
| h | #{k ≥ 1 : k/W < 1/H} | **ψ₅** (el mismo que en la lámina) |
| t | #{(k, l) ≠ 0 : (k/W)² + (l/H)² < 1/T²} | **ψ₁₂₉** |

Para ver el grosor, DMAP necesita **130 modos**. Además, la curvatura acopla t con s, así que la información del grosor no queda en un solo modo: se reparte en una **banda** de modos alrededor de ψ₁₂₉, y la parte A la mide. Es el caso extremo de «DMAP no es reducción de dimensión»: la dimensión intrínseca es 3, pero las tres coordenadas no aparecen hasta el modo 130.

| | Pregunta | Diseño |
|---|---|---|
| **A** | ¿Se cumple la predicción? ¿Cómo se comparan DMAP, IMAP y UMAP? | lectura afín (oráculo) de (s, h, t) desde d coordenadas, d = 1 … 512, error **por eje** normalizado (1 = sin información); posición medida del primer modo de h y de t; espectro de la lectura. 3 réplicas, N = 8000, sin entrenamiento |
| **B** | ¿Un autoencoder con latente 3-D recupera la carta? | DMAP + AE (continuación, λ = 0 y jac 0.1), AE sin DMAP (R³ → R³ → R³) y AE + oráculo (fijo y libre), m = 4 … 256; 6 réplicas |

Duración estimada con 2 × T4: β ~15 min, A ~30 min, B ~2.5 h.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json', 'latent_figure.py', 'export_figs.py', 'volume.py']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_12.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_12.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## Geometría y predicción

Cambia `T` para otro grosor; la predicción se recalcula. Para T = 2.5 el modo de t pasa a ψ₈₄, pero las vueltas quedan a 1.27, más cerca del ancho del núcleo.

In [ ]:
import importlib, volume as V
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
import protocol as P
W, H, T, N = 60., 10., 2., 8000
TEO = V.expected_positions(W, H, T)
lam_box, klj = V.box_spectrum(W, H, T, 300)
print(f"predicción: primer modo de h = psi_{TEO['h_mode']}, primer modo de t = psi_{TEO['t_mode']}")
print('modos de la caja alrededor del de t:', [tuple(map(int, q)) for q in klj[TEO['t_mode']-2:TEO['t_mode']+3]])
DATA = dict(dataset='swissvol', n=N, width=W, height=H, thickness=T)

## β

In [ ]:
# beta con la misma regla sin etiquetas del Swiss roll (masa de cortocircuito del grafo kNN), para esta geometría
cfg_v = json.loads((ROOT / 'config_beta_sweep.json').read_text())
cfg_v.update(DATA, data_seeds=[1000, 1001, 1002], factors=[16, 24, 32, 48, 64, 96, 128, 192],
             modes=[8, 64, 128, 256], out='/kaggle/working/beta_sweep_swissvol')
found = sorted(Path('/kaggle/input').rglob('beta_sweep_swissvol/beta_selection.json'))
sel_v = json.loads(found[0].read_text()) if found else P.run_sweep(cfg_v, ROOT, 'beta_sweep_swissvol')
FACTOR_V = sel_v['selected_factor'] or 64
if sel_v.get('edge_warning'):
    print('AVISO:', sel_v['edge_warning'])
print(f"factor beta (volumen) = {FACTOR_V}; admisibles {sel_v.get('admissible')}")

## A · Lectura afín de (s, h, t) frente a d (sin entrenamiento)

Igual que la Figura 2 del artículo, pero error **por eje** normalizado por su varianza. Los embeddings se calculan sobre los N puntos (en la muestra); β es el del barrido. IMAP es el escalamiento clásico de las geodésicas del grafo kNN. UMAP es opcional y solo hasta `UMAP_DMAX`, porque con más componentes es lento.

In [ ]:
import subprocess as _sp
UMAP_DMAX = 32
if UMAP_DMAX:
    _sp.run([sys.executable, '-m', 'pip', 'install', '-q', 'umap-learn'], check=False)
SCAN = [1, 2, 3, 4, 5, 6, 7, 8, 16, 32, 64, 96, 120, 128, 132, 140, 160, 192, 256, 384, 512]
SEEDS_A = [1000, 1001, 1002]                 # = réplicas 0, 1, 2 de la parte B
fA = Path('/kaggle/working/vol_readout.csv')
prev = sorted(Path('/kaggle/input').rglob('vol_readout.csv'))
if prev and not fA.exists():
    import shutil; shutil.copy(prev[0], fA)
RA, EXA = (pd.read_csv(fA), {}) if fA.exists() else V.readout_scan(DATA, SEEDS_A, SCAN, FACTOR_V, ('dmap', 'imap'), UMAP_DMAX)
RA.to_csv(fA, index=False)
POS = pd.DataFrame([dict(data_seed=s, **e['positions']) for s, e in EXA.items() if 'positions' in e])
if len(POS):
    POS.to_csv('/kaggle/working/vol_mode_positions.csv', index=False)
    display(POS.assign(h_pred=TEO['h_mode'], t_pred=TEO['t_mode']).style.format(precision=3)
            .set_caption('Qué modos llevan cada eje: primer modo con > 1 % de la varianza, modo con la mayor fracción y d con error < 0.5; predicción de la caja plana'))
med = RA.groupby(['method', 'd'])[['rel', 's', 'h', 't']].median()
display(med.unstack(0).style.format('{:.3g}').set_caption('Medianas sobre réplicas'))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))
colores = {'dmap': 'C0', 'imap': 'C1', 'umap': 'C3'}
for ax, col, titulo in zip(axes, ['rel', 's', 'h', 't'], ['‖Q − Q̂‖²_F / ‖Q − Q̄‖²_F', 'eje s', 'eje h', 'eje t (grosor)']):
    for m, g in RA.groupby('method'):
        q = g.groupby('d')[col]
        ax.plot(q.median().index, q.median().values, 'o-', ms=3, color=colores[m], label=m.upper())
        ax.fill_between(q.median().index, q.min().values, q.max().values, color=colores[m], alpha=.15, lw=0)
    for k, ls in (('h_mode', ':'), ('t_mode', '--')):
        ax.axvline(TEO[k]+1, color='k', ls=ls, lw=.8)      # el modo psi_k entra cuando d = k + 1
    ax.set(xscale='log', yscale='log', xlabel='d', title=titulo)
    if col != 'rel':
        ax.axhline(1, color='0.6', lw=.8, ls=':')
axes[0].legend(); fig.suptitle('Lectura afín (en la muestra). Verticales: d en que entra el primer modo de h (punteada) y de t (discontinua), predicho')
fig.tight_layout(); fig.savefig('/kaggle/working/vol_readout.png', dpi=200); fig.savefig('/kaggle/working/vol_readout.pdf'); plt.show()

# espectro de la lectura DMAP con d = max(SCAN) y fracción de la varianza de cada eje que lleva cada modo
if EXA:
    e = EXA[SEEDS_A[1]]
    L, lam = e['L'], e['lam'][:len(e['L'])]
    rate = -np.log(np.clip(lam, 1e-300, None))
    spec = pd.DataFrame({'n': np.arange(len(L)), 'rate': rate,
                         **{a: np.abs(L[:, i])/np.abs(L[:, i]).max() for i, a in enumerate(V.AXES)},
                         **{f'share_{a}': e['shares'][a][:len(L)] for a in V.AXES}})
    spec.to_csv('/kaggle/working/vol_spectrum.csv', index=False)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 4.2))
    for a, c in zip(V.AXES, ('C0', 'C1', 'C2')):
        a1.plot(spec.rate, spec[a], color=c, lw=.7, label=f'|L| eje {a}')
    a1.set(xscale='log', yscale='log', xlabel='−log λ_n', title='Espectro de la lectura DMAP (normalizado por eje)'); a1.legend()
    for a, c in zip(V.AXES, ('C0', 'C1', 'C2')):
        a2.plot(spec.n, spec[f'share_{a}'], color=c, lw=.8, label=f'fracción de Var({a}) del modo n')
    a2.set_yscale('log')
    a2.axvline(TEO['h_mode'], color='k', ls=':', lw=.8); a2.axvline(TEO['t_mode'], color='k', ls='--', lw=.8)
    a2.set(xlabel='índice del modo n', title='¿Qué modos llevan cada eje? (verticales: predicción)'); a2.legend()
    fig.tight_layout(); fig.savefig('/kaggle/working/vol_spectrum.png', dpi=200); plt.show()

### A · Carta verdadera frente a la predicha (como la Figura 3, en 3-D)

Cada método tiene dos filas, las proyecciones (s, h) y (s, t), y la primera columna es la verdad. Las demás columnas son la reconstrucción afín Q̂ con d coordenadas. Color = s verdadera en (s, h) y t verdadera en (s, t) (escala plasma): si el grosor no se recupera, los colores aparecen mezclados. En la esquina, el error por eje dividido por su varianza (s, h, t). En (s, t) el grosor está muy ampliado (60 contra 2) para que se vea si t se recupera: mientras DMAP no tenga el modo del grosor, (s, t) es una banda sin estructura en t.

In [ ]:
if not EXA:   # parte A reutilizada desde un CSV: recalcula solo la réplica de las figuras
    _, EXA = V.readout_scan(DATA, [SEEDS_A[1]], sorted(set(V.SHOW_D) | {3}), FACTOR_V, ('dmap', 'imap'), UMAP_DMAX)
V.readout_grid(EXA[SEEDS_A[1]], ['dmap', 'imap', 'umap'], '/kaggle/working/vol_verdad_vs_prediccion',
               title=f'Swiss roll con volumen (T = {T:g}): verdad y lectura afín de (s, h, t) desde d coordenadas · réplica {SEEDS_A[1] - 1000}')
plt.show()

## B · Autoencoders con latente 3-D

Mismo protocolo del 08 v2 (10⁴ pasos, Adam 10⁻³, lote 512, selección por validación, réplicas con muestra y partición nuevas), con `latent = 3`. La escalera de m cruza las dos predicciones: entre 4 y 8 entra h, y entre 128 y 160 entra t. Como N es mayor que en la lámina, se fijan `epochs = 1000`, que con 4800 puntos de entrenamiento son los mismos 10⁴ pasos.

In [ ]:
BASE = json.loads((ROOT / 'config_swiss.json').read_text())
BASE.update(DATA, latent=3)
COMMON = dict(P.COMMON, epochs=1000)
LADDER = [4, 8, 64, 128, 160, 256]
M = max(LADDER)
SEEDS_B = list(range(6))
comun = dict(beta_factor=FACTOR_V, modes=LADDER, m_protocols=['scratch'], mode_order='spectral',
             oracle_decoder=False, reg_k=10, latent=3)
def entrada(tag, models, lam=0., **extra):
    p = dict(comun, tag=tag, models=models, seeds=SEEDS_B, out=f'/kaggle/working/results_swissvol_{tag}')
    p.update(extra)
    if lam > 0:
        p.update(reg='jac', reg_weight=lam)
    return p
PLAN = [entrada('AE_l0', ['ae'], m_protocols=['scratch', 'warm'], scratch_first_only=True),
        entrada('AE_l0.1', ['ae'], 0.1, m_protocols=['scratch', 'warm'], scratch_first_only=True),
        entrada('PLAIN_AE_l0', ['plainae']),
        entrada('PLAIN_AE_l0.1', ['plainae'], 0.1),
        entrada('ORACLE_l0', ['oracle_ae', 'oracle_ae_free'], oracle_decoder=True)]
P.reuse_inputs(PLAN)
OUT = {p['tag']: Path(p['out']) for p in PLAN}
SECONDS_PER_MODEL = 80 * 1.4
pendientes = [p for p in PLAN if not (Path(p['out']) / 'metrics.csv').exists()]
P.show_plan(pendientes, 'swissvol', SECONDS_PER_MODEL, COMMON) if pendientes else print('todo reutilizado')

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(pendientes, BASE, ROOT, common=COMMON)

### Tabla: latente 3-D frente al oráculo, por eje

Latente alineado por semejanza (rotación o reflexión 3-D más una escala, Procrustes). Para cada eje, error / Var(eje); IC 95 % por bootstrap de la mediana. Se añaden la **tasa de despliegue** (regresión kNN latente → carta dentro de 10 veces el piso, invariante a reparametrizaciones suaves) y ‖x − x̂‖².

In [ ]:
rng_b = np.random.default_rng(0)
def mediana_ic(x, n=5000):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if not len(x):
        return np.nan, np.nan, np.nan
    b = np.median(rng_b.choice(x, (n, len(x))), axis=1)
    return np.median(x), *np.quantile(b, [.025, .975])
ETIQ = {('AE_l0', 'ae_warm'): 'volumen · DMAP + AE cont.', ('AE_l0.1', 'ae_warm'): 'volumen · DMAP + AE cont. + jac 0.1',
        ('PLAIN_AE_l0', 'plainae'): 'volumen · AE sin DMAP', ('PLAIN_AE_l0.1', 'plainae'): 'volumen · AE sin DMAP + jac 0.1',
        ('ORACLE_l0', 'oracle_ae'): 'volumen · DMAP + oráculo (encoder fijo)',
        ('ORACLE_l0', 'oracle_ae_free'): 'volumen · DMAP + AE iniciado en el oráculo'}
orden = list(ETIQ.values())
LT, AMB = [], []
for (tag, model), lab in ETIQ.items():
    d = P.load(OUT[tag])
    if d is None:
        print('falta', tag); continue
    t = V.latent_vs_oracle_table_3d(OUT[tag], [model])
    if len(t):
        LT.append(t.assign(corrida=lab, tag=tag, modelo=model))
    q = P.with_chain_start(d)
    AMB.append(q[q.model == model][['seed', 'm', 'mse_x', 'unfolded', 'knn_chart_ratio', 'local_distortion']].assign(corrida=lab, tag=tag, modelo=model))
LT, AMB = pd.concat(LT), pd.concat(AMB)
COLS = [(f'latent_vs_oracle_{a}_norm', f'lat↔oráculo {a}') for a in V.AXES] + \
       [(f'latent_vs_truth_{a}_norm', f'lat↔verdad {a}') for a in V.AXES] + \
       [(f'oracle_vs_truth_{a}_norm', f'oráculo↔verdad {a}') for a in V.AXES]
filas = []
for (lab, m), g in LT.groupby(['corrida', 'm']):
    fila = dict(corrida=lab, m=int(m), n=g.seed.nunique())
    for c, nombre in COLS:
        med, lo, hi = mediana_ic(g[c])
        fila[nombre] = med; fila[nombre + ' IC'] = f'{lo:.3g}–{hi:.3g}' if np.isfinite(lo) else ''
    a = AMB[AMB.corrida == lab]
    a = a[a.m == (a.m.min() if 'sin DMAP' in lab else m)]
    med, lo, hi = mediana_ic(a.mse_x)
    fila['‖x−x̂‖²'] = med; fila['‖x−x̂‖² IC'] = f'{lo:.3g}–{hi:.3g}'
    fila['desplegadas'] = f"{int(a.unfolded.astype(bool).sum())}/{len(a)}"
    filas.append(fila)
TV = pd.DataFrame(filas)
TV = TV.iloc[sorted(range(len(TV)), key=lambda i: (orden.index(TV.corrida.iloc[i]), TV.m.iloc[i]))].reset_index(drop=True)
TV.to_csv('/kaggle/working/vol_tabla_normalizada.csv', index=False)
num = [n for _, n in COLS] + ['‖x−x̂‖²']
display(TV[TV.m == M][['corrida', 'n', 'desplegadas'] + [c for n in num for c in (n, n + ' IC')]]
        .style.format({n: '{:.3g}' for n in num}, na_rep='—').set_caption(f'm = {M}'))
d0 = P.load(OUT['ORACLE_l0'])
if d0 is not None:
    REFV = pd.DataFrame([dict(m=m, oraculo_decoder_verdadero=d0[(d0.model == 'oracle_decoder') & (d0.m == m)].mse_x.median(),
                              decoder_carta_verdadera=d0[d0.model == 'chart_decoder'].mse_x.median()) for m in LADDER])
    REFV.to_csv('/kaggle/working/vol_referencias_ambiente.csv', index=False)
    display(REFV.style.format('{:.4g}').set_caption('Referencias del error ambiental'))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4.4))
est = {'ae_warm': 'o', 'plainae': None, 'oracle_ae': 's', 'oracle_ae_free': 'D'}
for i, ((tag, model), lab) in enumerate(ETIQ.items()):
    c, ls = f'C{i}', ':' if tag.endswith('0.1') else '-'
    g = AMB[(AMB.tag == tag) & (AMB.modelo == model)].groupby('m').mse_x.median()
    if model == 'plainae':
        axes[0].axhline(g.iloc[0], color=c, ls='--', label=lab)
    else:
        axes[0].plot(g.index, g.values, color=c, marker=est[model], ls=ls, label=lab)
    for ax, a in zip(axes[1:], V.AXES):
        l = LT[(LT.tag == tag) & (LT.modelo == model)].groupby('m')[f'latent_vs_oracle_{a}_norm'].median()
        ax.plot(l.index, np.maximum(l.values, 1e-12), color=c, marker=est[model] or '.', ls=ls if model != 'plainae' else '--', label=lab)
ref = LT[(LT.tag == 'AE_l0')].groupby('m')
for ax, a in zip(axes[1:], V.AXES):
    o = ref[f'oracle_vs_truth_{a}_norm'].median()
    ax.plot(o.index, o.values, 'k--', lw=1, label='oráculo vs verdad')
    ax.axhline(1, color='0.6', lw=.8, ls=':')
    for k, lsv in (('h_mode', ':'), ('t_mode', '--')):
        ax.axvline(TEO[k]+1, color='k', ls=lsv, lw=.6)
    ax.set(title=f'latente vs oráculo: eje {a}', ylabel=f'error / Var({a})')
axes[0].set(title='‖x − x̂‖²', ylabel='MSE por coordenada (test)')
for ax in axes:
    ax.set(xscale='log', yscale='log', xlabel='m (modos DMAP)')
    ax.set_xticks(LADDER); ax.set_xticklabels([str(m) for m in LADDER]); ax.minorticks_off()
axes[0].legend(fontsize=6); axes[3].legend(fontsize=6)
fig.tight_layout(); fig.savefig('/kaggle/working/vol_curvas.png', dpi=200); fig.savefig('/kaggle/working/vol_curvas.pdf'); plt.show()

### B · Latente predicho frente a la carta verdadera, y anillos

**Latente (como `final_latente`).** El latente 3-D de cada modelo se lleva a la carta con una afín ajustada en los puntos de entrenamiento y se dibuja sobre los de test, en las proyecciones (s, h) (color = s) y (s, t) (color = t). La fila «DMAP (lectura afín)» es el oráculo. Se usa la réplica mediana de cada modelo en m = 256.

**Anillos.** Puntos del oráculo (círculos) y latente alineado a ellos por semejanza (rotación o reflexión 3-D más una escala; puntos), unidos por segmentos grises. En las esquinas, el error por eje dividido por su varianza.

In [ ]:
filas = [dict(label='DMAP (lectura afín)', folder=OUT['AE_l0'], model='oracle')] + \
        [dict(label=lab.replace('volumen · ', ''), folder=OUT[tag], model=model) for (tag, model), lab in ETIQ.items() if P.load(OUT[tag]) is not None]
V.latent_grid_3d(filas, LADDER, '/kaggle/working/vol_latente', 'Swiss roll con volumen: latente 3-D llevado a la carta (afín, entrenamiento) frente a la verdad (test)')
plt.show()
V.rings_3d(filas[1:], LADDER, '/kaggle/working/vol_anillos', 'Puntos del oráculo (círculos) y latente alineado por semejanza (puntos)')
plt.show()

## Exportar para el artículo

`paper_figs_vol.zip` contiene:
- **tablas**: `vol_readout.csv`, `vol_mode_positions.csv`, `vol_spectrum.csv`, `vol_tabla_normalizada.csv` y `vol_referencias_ambiente.csv`;
- **paneles PNG transparentes** de la réplica 1, solo puntos (color = s en (s, h) y t en (s, t)): la verdad y las reconstrucciones afines de DMAP e IMAP para d = 8, 64, 128, 256 y 512, en las proyecciones (s, h) y (s, t);
- **para los AE**: paneles del latente 3-D llevado a la carta, réplica 1, todas las m y las dos proyecciones (`vol_lat_<corrida>_<modelo>_m<m>_<sh|st>.png`, más la verdad).

In [ ]:
import shutil
dest = Path('/kaggle/working/paper_figs_vol'); (dest / 'img').mkdir(parents=True, exist_ok=True)
EXT = {'sh': (-3.6, 63.6, -2.5, 12.5), 'st': (-3.6, 63.6, -T/2-.6, T/2+.6)}
def proyeccion(Qx, k):
    return Qx[:, [0, 1]] if k == 'sh' else Qx[:, [0, 2]]
def color(Qt, k):   # (s, h): colour = s (viridis); (s, t): colour = t (plasma)
    return dict(color=Qt[:, 0]) if k == 'sh' else dict(color=Qt[:, 2], vmin=-T/2, vmax=T/2, cmap='plasma')
if EXA:
    e = EXA[SEEDS_A[1]]
    Q = e['Q']
    for k in EXT:
        V.panel(dest / 'img' / f'vol_truth_{k}.png', proyeccion(Q, k), extent=EXT[k], s=.5, **color(Q, k))
        for key, Qh in e.items():
            if key.startswith('recon_'):
                V.panel(dest / 'img' / f"vol_{key[6:]}_{k}.png", proyeccion(Qh, k), extent=EXT[k], s=.5, **color(Q, k))
for (tag, model), lab in [(('AE_l0', 'oracle'), 'oráculo')] + list(ETIQ.items()):
    f = OUT[tag]
    if P.load(f) is None or not (f / 'seed_1').exists():
        continue
    for m in LADDER:
        z, t = V.latent_3d(f, 1, model, m, LADDER[0])
        if z is None:
            continue
        for k in EXT:
            V.panel(dest / 'img' / f'vol_lat_{tag}_{model}_m{m}_{k}.png', proyeccion(z, k), extent=EXT[k], s=1., **color(t, k))
            if model == 'oracle' and m == LADDER[0]:
                V.panel(dest / 'img' / f'vol_lat_truth_{k}.png', proyeccion(t, k), extent=EXT[k], s=1., **color(t, k))
for fname in ('vol_readout.csv', 'vol_mode_positions.csv', 'vol_spectrum.csv', 'vol_tabla_normalizada.csv', 'vol_referencias_ambiente.csv'):
    if Path('/kaggle/working', fname).exists():
        shutil.copy(Path('/kaggle/working', fname), dest / fname)
(dest / 'geometria.json').write_text(json.dumps(dict(W=W, H=H, T=T, N=N, beta_factor=FACTOR_V, prediccion=TEO), indent=1))
shutil.make_archive('/kaggle/working/paper_figs_vol', 'zip', dest)
print(sorted(p.name for p in dest.rglob('*')))

## Qué descargar

**`paper_figs_vol.zip`**: con las tablas y los paneles basta para el artículo. Para reutilizar sin reentrenar, guarda también `results_swissvol_*` y `beta_sweep_swissvol/`; `vol_readout.csv` evita repetir la parte A.